# 08 - SYSI Synthetic Soil Image (GEOS3 bare-soil composite)

**What:** Build a SYSI (SYnthetic Soil Image) bare-soil composite from Sentinel-2. The pipeline cloud-masks the S2 SR Harmonized collection, computes spectral indices, applies the **GEOS3** bare-soil detection rule (Geospatial Soil Sensing System), keeps only the bare-soil pixels of each scene, and takes the **median** of those pixels through time. The result is a clean soil reflectance image with no vegetation/crop cover.

**Why:** Agronomists need the spectral signature of the *soil itself* (texture, organic matter, mineralogy proxies) without the noise of whatever crop happened to be growing on a given date. A single date rarely has bare soil over the whole field, but across a multi-year window every pixel is bare at *some* point (between harvest and the next sowing). GEOS3 detects those bare-soil moments per pixel and the temporal median stitches them into one synthetic, gap-free soil image.

**GEOS3 rule (bare soil) — a pixel is soil when ALL hold:**
- NDVI within `ndvi_thres` (low vegetation),
- NBR2 within `nbr_thres` (limits crop residue / non-photosynthetic vegetation),
- VNSIR trend `<= vnsir_thres` (visible-to-shortwave-infrared slope typical of soil),
- GRBL `> 0` (Green > Blue) and REGR `> 0` (Red > Green) — the rising visible slope characteristic of soil rather than green canopy or water.

**Legacy reference (logic copied exactly, not reinvented):**
- `legacy:ravi_dialog.py` -> `soil_image` (~L4636-4860): `maskS2clouds`, `add_indexes` (NDVI, NBR2, GRBL, REGR), `applyScaleFactors`, `addGEOS3Mask` (NDVI/NBR2/VNSIR/GRBL/REGR thresholds), `maskByGEOS3`, month filtering, `COPERNICUS/S2_SR_HARMONIZED` loading, `s2_names`/`b_names`, `bands_to_export` + `.median()`.
- `legacy:ravi_dialog.py` -> `sysi_processing` (~L4860-4900): the final `toFloat()` + AOI mask used for download.
- `legacy:modules/sysi.py`: collection loader stub.

In [ ]:
# --- Setup ---
import os
import ee
import pandas as pd
from IPython.display import Image, display

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)
print("Earth Engine initialized.")

In [ ]:
# --- AOI + dates + GEOS3 parameters ---
# AOI loaded from the project shapefile (same area as dev/testing.ipynb).
# Legacy uses self.aoi.geometry().bounds(); aoi2 is the bounding box used for filterBounds.
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


aoi = load_aoi_from_shapefile("contorno_area_total.zip").geometry()
aoi2 = aoi.bounds()

# A multi-year window maximizes the chance that every pixel is bare soil at some point.
START_DATE = "2019-01-01"
END_DATE = "2023-12-31"

# --- Parameters (defaults match the legacy sliders) ---
cloud_threshold = 10            # CLOUDY_PIXEL_PERCENTAGE upper bound (legacy horizontalSlider_clouds)
ndvi_thres = [-0.25, 0.25]      # NDVI band for bare soil (legacy ndvi_inf/ndvi_sup sliders /100)
nbr_thres = [-0.30, 0.10]       # NBR2 band for bare soil (legacy nbr_inf/nbr_sup sliders /100)
vnsir_thres = 0.9               # VNSIR trend ceiling (fixed in legacy)
selected_months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]  # legacy month checkBox_1..12

# Band name mapping (legacy s2_names -> b_names). Only these S2 bands are carried through.
s2_names = ['B2', 'B3', 'B4', 'B6', 'B8', 'B11', 'B12', 'QA60']
b_names = ['Blue', 'Green', 'Red', 'Rededge2', 'NIR', 'SWIR1', 'SWIR2', 'QA60']

print("AOI bounds:", aoi2.getInfo()['coordinates'])
print("Date range:", START_DATE, "->", END_DATE)
print("cloud_threshold:", cloud_threshold, "| ndvi_thres:", ndvi_thres, "| nbr_thres:", nbr_thres, "| vnsir_thres:", vnsir_thres)
print("selected_months:", selected_months)

## `maskS2clouds` — QA60 cloud / cirrus mask

Bits 10 (opaque clouds) and 11 (cirrus) of the `QA60` band must both be zero. Exact port of the legacy function.

In [ ]:
def maskS2clouds(image):
    qa = image.select('QA60')
    # Bits 10 e 11 sao nuvens e cirrus
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    # Ambos devem ser zero
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask)

## `add_indexes` — NDVI, NBR2, GRBL, REGR

- **NDVI** = (NIR - Red)/(NIR + Red): vegetation greenness.
- **NBR2** = (SWIR1 - SWIR2)/(SWIR1 + SWIR2): sensitive to crop residue / moisture.
- **GRBL** = Green - Blue: positive over soil (rising visible slope).
- **REGR** = Red - Green: positive over soil (continues the rising visible slope).

Exact port of the legacy function.

In [ ]:
def add_indexes(img):
    ndvi = img.normalizedDifference(['NIR', 'Red'])
    nbr = img.normalizedDifference(['SWIR1', 'SWIR2'])
    grbl = img.select('Green').subtract(img.select('Blue'))
    regr = img.select('Red').subtract(img.select('Green'))

    img = img.addBands(ndvi.rename('NDVI')) \
             .addBands(nbr.rename('NBR2')) \
             .addBands(grbl.rename('GRBL')) \
             .addBands(regr.rename('REGR'))
    return img

## `applyScaleFactors` — Sentinel-2 reflectance scaling

Divides the optical bands (`b_names`) by 10000 to convert S2 SR digital numbers into surface reflectance. The `overwrite=True` flag replaces the original bands in place. Exact port of the legacy function.

In [ ]:
def applyScaleFactors(image):
    opticalBands = image.select(b_names).divide(10000)
    return image.addBands(opticalBands, None, True)

## `addGEOS3Mask` — the GEOS3 bare-soil detector

Computes the **VNSIR** visible-to-shortwave-infrared trend and combines it with the index thresholds into the boolean `GEOS3` band (1 = bare soil):

`VNSIR = 1 - ( 2*Red - Green - Blue + 3*(NIR - Red) )`

`GEOS3 = (ndvi_thres[0] <= NDVI <= ndvi_thres[1]) AND (nbr_thres[0] <= NBR2 <= nbr_thres[1]) AND (VNSIR <= vnsir_thres) AND (GRBL > 0) AND (REGR > 0)`

Exact port of the legacy function (options dict preserved).

In [ ]:
def addGEOS3Mask(img, options):
    # Extracao de opcoes do dicionario
    rescale_flag = options.get('rescale_flag', 0)
    ndvi_thres = options.get('ndvi_thres', [-0.25, 0.25])
    nbr_thres = options.get('nbr_thres', [-0.3, 0.1])
    vnsir_thres = options.get('vnsir_thres', 0.9)

    if rescale_flag == 1:
        img = img.divide(10000)

    # Indice de tendencia Visible-to-shortwave-infrared
    vnsir = ee.Image(1).subtract(
        ee.Image(2).multiply(img.select('Red'))
        .subtract(img.select('Green')).subtract(img.select('Blue'))
        .add(ee.Image(3).multiply(img.select('NIR').subtract(img.select('Red'))))
    )

    # Logica GEOS3
    geos3 = img.select('NDVI').gte(ndvi_thres[0]).And(img.select('NDVI').lte(ndvi_thres[1])) \
        .And(img.select('NBR2').gte(nbr_thres[0]).And(img.select('NBR2').lte(nbr_thres[1]))) \
        .And(vnsir.lte(vnsir_thres)) \
        .And(img.select('GRBL').gt(0)).And(img.select('REGR').gt(0))

    img = img.addBands(geos3.rename('GEOS3'))
    return img

## `maskByGEOS3` — keep only bare-soil pixels

Masks each scene to where `GEOS3 == 1`, plus drops black borders / no-data edges by requiring SWIR2, Green, Red and Blue to be `>= 0`. Exact port of the legacy function.

In [ ]:
def maskByGEOS3(image):
    # Cria mascara onde GEOS3 e verdadeiro (1)
    mask_geos3 = image.select('GEOS3').eq(1)
    # Removendo linhas pretas/bordas
    mask_swir = image.select('SWIR2').gte(0)
    mask_green = image.select('Green').gte(0)
    mask_red = image.select('Red').gte(0)
    mask_blue = image.select('Blue').gte(0)

    mask = mask_geos3.And(mask_swir).And(mask_green).And(mask_red).And(mask_blue)
    return image.updateMask(mask)

## Build the collection and run the SYSI pipeline

Mirrors the legacy main flow in `soil_image`:
1. Load `COPERNICUS/S2_SR_HARMONIZED`, filter by date / bounds / cloud %, cloud-mask, rename bands.
2. Tag each image with its month and keep only `selected_months`.
3. Map `add_indexes` then `applyScaleFactors`.
4. Map `addGEOS3Mask` with the threshold options, then `maskByGEOS3`.
5. Take the temporal **median** of the bare-soil pixels and select `bands_to_export`.

In [ ]:
# 1. Carregar Colecao e Filtragem Inicial
dataset_s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate(START_DATE, END_DATE) \
    .filterBounds(aoi2) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold)) \
    .map(maskS2clouds) \
    .select(s2_names, b_names)

print("Scenes after date/bounds/cloud filter:", dataset_s2.size().getInfo())

# Filter collection by selected months (1 = Jan, 12 = Dec)
if len(selected_months) == 0:
    raise ValueError("Please select at least one month.")

def _add_month(image):
    month = ee.Date(image.get('system:time_start')).get('month')
    return image.set('month', month)

dataset_s2 = dataset_s2.map(_add_month)
dataset_s2 = dataset_s2.filter(ee.Filter.inList('month', ee.List(selected_months)))
print("Scenes after month filter:", dataset_s2.size().getInfo())

# 2. Adicionar Indices e Fatores de Escala
s2_indexes = dataset_s2.map(add_indexes)
s2_indexes = s2_indexes.map(applyScaleFactors)

# 3. Aplicacao da Mascara GEOS3 na colecao
geos3_options = {
    'rescale_flag': 0,
    'ndvi_thres': ndvi_thres,
    'nbr_thres': nbr_thres,
    'vnsir_thres': vnsir_thres,
}
sent_collection_geos3 = s2_indexes.map(lambda img: addGEOS3Mask(img, geos3_options))

# 4. Obter a mediana dos pixels de solo exposto (TESS / SYSI)
tess_v1_collection = sent_collection_geos3.map(maskByGEOS3)
tess_v1 = tess_v1_collection.median()

# 5. Seleciona as bandas finais
bands_to_export = ['Blue', 'Green', 'Red', 'Rededge2', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'NBR2']
sysi = tess_v1.select(bands_to_export).toFloat().clip(aoi)

print("SYSI bands:", sysi.bandNames().getInfo())

## Verify: per-band mean over the AOI

In [ ]:
# reduceRegion mean over the AOI (legacy calculateMean uses scale=10).
stats = sysi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi,
    scale=10,
    maxPixels=1e9,
).getInfo()

means = pd.DataFrame(
    [{"band": b, "mean": stats.get(b)} for b in bands_to_export]
)
print("SYSI mean reflectance / index over AOI:")
print(means.to_string(index=False))

## Verify: true-color thumbnail of the SYSI soil image

Renders Red/Green/Blue of the bare-soil composite. Reflectance is already scaled to 0-1, so the visualization range is `min=0, max=0.3` (typical soil reflectance).

In [ ]:
thumb_url = sysi.getThumbURL({
    'bands': ['Red', 'Green', 'Blue'],
    'min': 0.0,
    'max': 0.3,
    'region': aoi2.getInfo()['coordinates'],
    'dimensions': 512,
    'format': 'png',
})
print("Thumbnail URL:", thumb_url)
display(Image(url=thumb_url))